# 16 · DPO, assembled

**The whole chain:**
- 13: a response has a **log-probability**; SFT just raises the gold one.
- 14: we get **(chosen, rejected)** pairs free, from the execution reward.
- 15: preference = `σ(score gap)`, and we minimize `−log σ(gap)`.

The last missing piece is the **score**. DPO's clever move is what it uses as the
score — and it needs **no reward model at all**.

In [ ]:
import numpy as np
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

## Step 1 — The implicit reward (DPO's key idea)

DPO scores a response by **how much more our current model likes it than a frozen
reference model does.** The reference (`π_ref`) is the model we started from — here,
the SFT-tuned model. The score (an *implicit reward*) is:

$$r(\text{response}) = \beta \,\big(\, \log \pi(\text{response}) - \log \pi_{\text{ref}}(\text{response}) \,\big)$$

In words: *"did we make this response more likely than the reference did?"* The
reference acts as a **leash** — we want to prefer good answers without drifting into
gibberish far from the sensible starting model. `β` sets how tight the leash is.

## Step 2 — Plug the reward gap into Bradley-Terry → the DPO loss

Put that score into notebook 15's `σ(score gap)` and take `−log`:

$$\mathcal{L} = -\log \sigma\Big( \beta\big[(\log\pi_c - \log\pi_{\text{ref},c}) - (\log\pi_r - \log\pi_{\text{ref},r})\big] \Big)$$

where `c` = chosen, `r` = rejected. Minimizing it makes the policy raise the chosen
log-prob and lower the rejected one — **relative to the reference**. Notice there is
no reward model anywhere: the reward is written purely from the policy's own
log-probabilities. *That* is why DPO is "direct."

## Step 3 — Watch it work

Start with the policy equal to the reference (margin 0 → loss `−log 0.5 ≈ 0.693`),
then nudge the log-probs to prefer chosen and watch the loss fall:

In [ ]:
beta = 0.1
ref_c, ref_r = -5.0, -5.0       # log-probs the FROZEN reference gives chosen / rejected
pol_c, pol_r = -5.0, -5.0       # our trainable policy, STARTING equal to the reference

def dpo_loss(pc, pr):
    margin = beta * ((pc - ref_c) - (pr - ref_r))   # the implicit-reward gap
    return -np.log(sigmoid(margin))

print("loss at start (policy = reference):", round(dpo_loss(pol_c, pol_r), 3))

lr = 0.5
for _ in range(60):
    margin = beta * ((pol_c - ref_c) - (pol_r - ref_r))
    g = (1 - sigmoid(margin)) * beta
    pol_c += lr * g             # make CHOSEN more likely
    pol_r -= lr * g             # make REJECTED less likely

print("after training:")
print("  log-prob chosen   :", round(pol_c, 2))
print("  log-prob rejected :", round(pol_r, 2))
print("  chosen preferred by:", round(pol_c - pol_r, 2), "in log-prob")
print("  final DPO loss     :", round(dpo_loss(pol_c, pol_r), 3))

The loss fell from ~0.69 toward 0 as the chosen–rejected margin grew — exactly what
DPO does, using only log-probs.

## Step 4 — How this maps to the repo

- `preference.train_dpo` (in `preference.py`) is this loss via `trl.DPOTrainer`.
- `ref_model=None` there means TRL uses the **frozen base/adapter as `π_ref`** — the
  leash from Step 1.
- `cfg.dpo_beta = 0.1` is the **`β`** above. Small `β` = loose leash (policy can move
  more); large `β` = stays close to the reference.
- The **(chosen, rejected)** pairs come from notebook 14 (`make_preference_pairs`).

## Recap — the objective axis

| | trains on | signal |
|---|---|---|
| **SFT** | one gold answer | imitate it (raise its log-prob) |
| **DPO** | (chosen, rejected) pairs | prefer chosen over rejected vs a frozen reference |

Same model, same PEFT mechanism, same execution-accuracy eval — only the **loss and
data shape** changed. That's the objective axis: DPO swaps *what* you train on, not
*how many* parameters.

**Triple BAM!** The remaining piece of the lab is **PPO** (online: sample, score with
the execution reward, update) — and then the *outcome* evidence: actually run
`--config dpo` on a GPU and compare execution accuracy against the SFT baseline.